In [6]:
# %%
import pandas as pd
import numpy as np
import glob
import os
from IPython.display import display

print("Step 1: Loading Raw Data...")

DATA_RAW = os.path.join("..", "data", "raw")
DATA_PROCESSED = os.path.join("..", "data", "processed")
os.makedirs(DATA_PROCESSED, exist_ok=True)

# Define exactly which columns to read from the massive PUMS files
h_cols = ['SERIALNO', 'PUMA', 'FS', 'FFSP', 'ACCESSINET', 'VEH', 'LNGI', 'GRPIP', 'RMSP', 'TEN', 'HINCP', 'NP']
p_cols = ['SERIALNO', 'POVPIP', 'AGEP', 'DIS', 'SCHL', 'ESR', 'RAC1P', 'HISP']

h_files = glob.glob(os.path.join(DATA_RAW, "psam_h*.csv"))
p_files = glob.glob(os.path.join(DATA_RAW, "psam_p*.csv"))

if not h_files or not p_files:
    raise FileNotFoundError("Missing PUMS files in data/raw. Please run the download script first.")

df_h_raw = pd.concat([pd.read_csv(f, usecols=h_cols) for f in h_files], ignore_index=True)
df_p_raw = pd.concat([pd.read_csv(f, usecols=p_cols) for f in p_files], ignore_index=True)

print(f"[SUCCESS] Loaded Households: {df_h_raw.shape}")
print(f"[SUCCESS] Loaded People: {df_p_raw.shape}")

Step 1: Loading Raw Data...
[SUCCESS] Loaded Households: (74024, 12)
[SUCCESS] Loaded People: (157694, 8)


In [7]:
# %%
print("Step 2: Engineering Person-Level Proxy Features...")

df_p = df_p_raw.copy()

# 1. Create binary flags for individuals
df_p['is_elderly'] = (df_p['AGEP'] >= 60).astype(int)
df_p['is_child'] = (df_p['AGEP'] < 18).astype(int)
df_p['is_disabled'] = (df_p['DIS'] == 1).astype(int) 
df_p['is_working'] = df_p['ESR'].isin([1, 2, 4, 5]).astype(int) 
df_p['is_minority'] = ((df_p['RAC1P'] != 1) | (df_p['HISP'] != 1)).astype(int) 
df_p['SCHL'] = df_p['SCHL'].fillna(0) 

# 2. Aggregate to Household Level (SERIALNO)
hh_agg = df_p.groupby('SERIALNO').agg(
    POVPIP=('POVPIP', 'first'),              # Crucial for defining the target later
    HAS_ELDERLY=('is_elderly', 'max'),       
    HAS_DISABLED=('is_disabled', 'max'),     
    NUM_CHILDREN=('is_child', 'sum'),        
    NUM_WORKING_ADULTS=('is_working', 'sum'),
    MAX_EDUCATION=('SCHL', 'max'),           
    IS_MINORITY_HH=('is_minority', 'max')    
).reset_index()

print("[SUCCESS] Aggregated person demographics to household level.")

Step 2: Engineering Person-Level Proxy Features...
[SUCCESS] Aggregated person demographics to household level.


In [8]:
# %%
print("Step 3: Merging, Filtering, and Creating the Blind Target...")

# 1. Merge
df_merged = pd.merge(df_h_raw, hh_agg, on='SERIALNO', how='inner')

# 2. Filter out Group Quarters and Imputed SNAP data
mask_standard = ~df_merged['SERIALNO'].astype(str).str.contains('GQ')
mask_real_snap = df_merged['FFSP'] != 1
df_clean = df_merged[mask_standard & mask_real_snap].copy()

# 3. Create Target Variable
# Class 1 (The Gap): Low income (<=130%) AND no SNAP (FS == 2)
# Class 0 (Not in Gap): Everyone else
df_clean['TARGET_GAP'] = ((df_clean['POVPIP'] <= 130) & (df_clean['FS'] == 2.0)).astype(int)

# 4. Handle Missing Values for socioeconomic proxy features
df_clean['VEH'] = df_clean['VEH'].fillna(0)  
df_clean['GRPIP'] = df_clean['GRPIP'].fillna(0) 
df_clean['ACCESSINET'] = df_clean['ACCESSINET'].fillna(3) 
df_clean['LNGI'] = df_clean['LNGI'].fillna(1) 
df_clean['TEN'] = df_clean['TEN'].fillna(0) 

# 5. FEATURE SELECTION (The "Blind" Step)
# We purposefully DROP POVPIP, HINCP, FS, and FFSP here.
final_features = [
    'SERIALNO', 'PUMA', 'TARGET_GAP', 'HAS_ELDERLY', 'HAS_DISABLED', 
    'NUM_CHILDREN', 'NUM_WORKING_ADULTS', 'MAX_EDUCATION', 'IS_MINORITY_HH',
    'VEH', 'ACCESSINET', 'LNGI', 'GRPIP', 'RMSP', 'TEN', 'NP'
]

df_final = df_clean[final_features].copy()

print(f"[SUCCESS] Final dataset ready: {df_final.shape[0]} total standard households.")
print(f"[SUCCESS] Target Distribution:\n{df_final['TARGET_GAP'].value_counts(normalize=True)*100}")

Step 3: Merging, Filtering, and Creating the Blind Target...
[SUCCESS] Final dataset ready: 61071 total standard households.
[SUCCESS] Target Distribution:
TARGET_GAP
0    91.929066
1     8.070934
Name: proportion, dtype: float64


In [9]:
# %%
print("Step 4: Generating Data Dictionary...")

doc_data = [
    {"Column": "SERIALNO", "Description": "Unique Household Identifier", "Type": "String"},
    {"Column": "PUMA", "Description": "Public Use Microdata Area (Geography Code)", "Type": "Categorical/Numeric"},
    {"Column": "TARGET_GAP", "Description": "TARGET: 1 if in the SNAP Gap, 0 otherwise", "Type": "Binary (0/1)"},
    {"Column": "HAS_ELDERLY", "Description": "1 if household has someone 60+", "Type": "Binary (0/1)"},
    {"Column": "HAS_DISABLED", "Description": "1 if household has someone with a disability", "Type": "Binary (0/1)"},
    {"Column": "NUM_CHILDREN", "Description": "Number of people under 18 in the household", "Type": "Numeric"},
    {"Column": "NUM_WORKING_ADULTS", "Description": "Number of employed adults in the household", "Type": "Numeric"},
    {"Column": "MAX_EDUCATION", "Description": "Highest educational attainment code in the household (1-24)", "Type": "Numeric (Ordinal)"},
    {"Column": "IS_MINORITY_HH", "Description": "1 if anyone in the household identifies as a racial/ethnic minority", "Type": "Binary (0/1)"},
    {"Column": "VEH", "Description": "Number of vehicles available (0-6)", "Type": "Numeric"},
    {"Column": "ACCESSINET", "Description": "Internet access: 1/2=Yes, 3=No", "Type": "Categorical"},
    {"Column": "LNGI", "Description": "Limited English speaking household: 1=Not limited, 2=Limited", "Type": "Categorical"},
    {"Column": "GRPIP", "Description": "Gross rent as a percentage of household income", "Type": "Numeric"},
    {"Column": "RMSP", "Description": "Number of rooms in the housing unit", "Type": "Numeric"},
    {"Column": "TEN", "Description": "Housing tenure (1/2=Own, 3/4=Rent)", "Type": "Categorical"},
    {"Column": "NP", "Description": "Number of person records in the household", "Type": "Numeric"}
]

df_documentation = pd.DataFrame(doc_data)
display(df_documentation)

Step 4: Generating Data Dictionary...


,Column,Description,Type
0,SERIALNO,Unique Household Identifier,String
1,PUMA,Public Use Microdata Area (Geography Code),Categorical/Numeric
2,TARGET_GAP,"TARGET: 1 if in the SNAP Gap, 0 otherwise",Binary (0/1)
3,HAS_ELDERLY,1 if household has someone 60+,Binary (0/1)
4,HAS_DISABLED,1 if household has someone with a disability,Binary (0/1)
5,NUM_CHILDREN,Number of people under 18 in the household,Numeric
6,NUM_WORKING_ADULTS,Number of employed adults in the household,Numeric
7,MAX_EDUCATION,Highest educational attainment code in the hou...,Numeric (Ordinal)
8,IS_MINORITY_HH,1 if anyone in the household identifies as a r...,Binary (0/1)
9,VEH,Number of vehicles available (0-6),Numeric


In [10]:
# %%
print("Step 5: Sampling, Displaying, and Saving Results...")

df_gap = df_final[df_final['TARGET_GAP'] == 1]      
df_non_gap = df_final[df_final['TARGET_GAP'] == 0] 

sample_gap = df_gap.sample(n=min(20, len(df_gap)), random_state=42)
sample_non_gap = df_non_gap.sample(n=min(20, len(df_non_gap)), random_state=42)

print("\nTHE GAP HOUSEHOLDS (TARGET_GAP = 1) - Sample of 20:")
display(sample_gap)

print("\nNON-GAP HOUSEHOLDS (TARGET_GAP = 0) - Sample of 20:")
display(sample_non_gap)

# Save processed data and dictionary
out_path = os.path.join(DATA_PROCESSED, "processed_all_households_blind.csv")
doc_path = os.path.join(DATA_PROCESSED, "data_dictionary_blind.csv")

df_final.to_csv(out_path, index=False)
df_documentation.to_csv(doc_path, index=False)

print(f"\n[SUCCESS] SUCCESS: Processed dataset saved to {out_path} ({df_final.shape[0]} rows)")
print(f"[SUCCESS] SUCCESS: Data Dictionary saved to {doc_path}")

Step 5: Sampling, Displaying, and Saving Results...

THE GAP HOUSEHOLDS (TARGET_GAP = 1) - Sample of 20:


,SERIALNO,PUMA,TARGET_GAP,HAS_ELDERLY,HAS_DISABLED,NUM_CHILDREN,NUM_WORKING_ADULTS,MAX_EDUCATION,IS_MINORITY_HH,VEH,ACCESSINET,LNGI,GRPIP,RMSP,TEN,NP
68858,2023HU1389337,7300,1,1,1,0,1,11.0,1,2.0,3.0,1.0,0.0,4.0,2.0,1
52089,2023HU0699551,71002,1,1,0,0,0,21.0,0,0.0,1.0,1.0,53.0,3.0,3.0,1
15488,2023HU0551209,601,1,0,0,0,0,20.0,0,2.0,1.0,1.0,0.0,5.0,1.0,1
61885,2023HU1100499,70000,1,0,1,0,0,19.0,0,1.0,1.0,1.0,0.0,8.0,1.0,2
53989,2023HU0775965,8702,1,1,0,0,0,24.0,0,2.0,1.0,1.0,0.0,9.0,2.0,2
66934,2023HU1308924,17100,1,0,0,0,1,16.0,0,2.0,1.0,1.0,0.0,4.0,1.0,1
46314,2023HU0453626,16500,1,0,1,0,0,20.0,0,0.0,2.0,1.0,101.0,1.0,3.0,1
20880,2023HU0869987,1107,1,1,0,0,0,13.0,0,1.0,1.0,1.0,0.0,7.0,4.0,1
64496,2023HU1206102,71001,1,0,0,1,1,19.0,0,1.0,1.0,1.0,0.0,9.0,1.0,2
1846,2023HU0568474,103,1,1,1,0,2,21.0,1,1.0,1.0,1.0,66.0,5.0,3.0,4



NON-GAP HOUSEHOLDS (TARGET_GAP = 0) - Sample of 20:


,SERIALNO,PUMA,TARGET_GAP,HAS_ELDERLY,HAS_DISABLED,NUM_CHILDREN,NUM_WORKING_ADULTS,MAX_EDUCATION,IS_MINORITY_HH,VEH,ACCESSINET,LNGI,GRPIP,RMSP,TEN,NP
52198,2023HU0704286,4103,0,1,1,0,1,21.0,0,3.0,1.0,1.0,0.0,9.0,2.0,2
23311,2023HU1016450,1102,0,0,0,1,3,21.0,1,4.0,1.0,1.0,0.0,8.0,1.0,6
68349,2023HU1368364,10701,0,0,0,0,3,21.0,0,4.0,1.0,1.0,0.0,9.0,1.0,4
65385,2023HU1245941,1301,0,0,0,3,2,21.0,0,2.0,1.0,1.0,0.0,19.0,1.0,5
3226,2023HU1198043,105,0,0,0,0,1,22.0,0,1.0,1.0,1.0,11.0,4.0,3.0,1
63999,2023HU1186938,8300,0,0,0,0,1,16.0,0,2.0,3.0,1.0,0.0,7.0,2.0,1
65072,2023HU1231210,6900,0,0,0,2,2,21.0,0,2.0,1.0,1.0,50.0,6.0,3.0,4
49445,2023HU0587943,5906,0,1,0,0,1,22.0,0,2.0,1.0,1.0,0.0,9.0,1.0,2
3161,2023HU1166248,102,0,0,0,0,3,22.0,0,2.0,1.0,1.0,23.0,4.0,3.0,3
44294,2023HU0368151,60001,0,0,0,0,2,22.0,0,4.0,1.0,1.0,0.0,9.0,1.0,2



[SUCCESS] SUCCESS: Processed dataset saved to ../data/processed/processed_all_households_blind.csv (61071 rows)
[SUCCESS] SUCCESS: Data Dictionary saved to ../data/processed/data_dictionary_blind.csv
